In [2]:
# 1. Import Library
import pandas as pd
import numpy as np
import os

# 2. Konfigurasi Path
INPUT_PATH = '../../kamus/inset_vader_political.csv' 
OUTPUT_PATH = '../../kamus/inset_vader_political_modified.csv'

np.random.seed(42)

# 3. Fungsi Adaptasi Skor (SLA)
def generate_vader_ratings(mean_val, std_val=1.0):
    """
    Menghasilkan 10 angka bulat yang rata-ratanya sesuai mean_val.
    Tetap menggunakan clip -4 sampai 4 karena ini adalah standar VADER.
    """
    target_sum = int(round(mean_val * 10))
    
    ratings = np.random.normal(loc=mean_val, scale=std_val, size=10)
    ratings = np.round(ratings).astype(int)
    # Clip ini tetap perlu agar tidak ada rater individu yang memberi skor diluar batas -4/4
    ratings = np.clip(ratings, -4, 4)
    
    current_sum = np.sum(ratings)
    diff = target_sum - current_sum
    
    if diff != 0:
        step = 1 if diff > 0 else -1
        indices = np.arange(10)
        while diff != 0:
            np.random.shuffle(indices)
            for idx in indices:
                new_val = ratings[idx] + step
                if -4 <= new_val <= 4:
                    ratings[idx] = new_val
                    diff -= step
                    if diff == 0: break
                    
    return ratings.tolist()

# 4. Eksekusi Restrukturisasi
df_politik = pd.read_csv(INPUT_PATH)
df_politik['kata'] = df_politik['kata'].astype(str).str.strip().str.lower()

df_modified = pd.DataFrame()
df_modified['kata'] = df_politik['kata']

# PERUBAHAN DISINI: 
# Langsung ambil skor dari file csv tanpa dikali 0.8 karena input sudah skala -4 s/d 4
df_modified['mean'] = df_politik['skor'].round(1) 

df_modified['std'] = 1.0
df_modified['raw_ratings'] = df_modified['mean'].apply(lambda m: generate_vader_ratings(m))

# 5. Simpan Hasil
df_modified.to_csv(OUTPUT_PATH, index=False)
print(f"[SUCCESS] Leksikon Politik berhasil direstrukturisasi ke: {OUTPUT_PATH}")

[SUCCESS] Leksikon Politik berhasil direstrukturisasi ke: ../../kamus/inset_vader_political_modified.csv
